# scRNA-seq PBMC Analysis
**Dataset:** 10x Genomics PBMC 3k  
**Stack:** Scanpy + AnnData  
**Pipeline:** QC → Preprocessing → Clustering → Annotation → Differential Expression

## 1. Setup

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)

print(f'Scanpy version: {sc.__version__}')

## 2. Load Data

In [ ]:
# Download and load the 10x PBMC 3k dataset (cached after first download)
adata = sc.datasets.pbmc3k()
print(adata)
print(f'\n{adata.n_obs} cells × {adata.n_vars} genes')

## 3. Quality Control

In [ ]:
# Flag mitochondrial genes (marker of cell stress / dying cells)
adata.var['mt'] = adata.var_names.str.startswith('MT-')

sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=['mt'],
    percent_top=None,
    log1p=False,
    inplace=True
)

adata.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sc.pl.violin(adata, 'n_genes_by_counts', jitter=0.4, ax=axes[0], show=False)
axes[0].set_title('Genes per cell')

sc.pl.violin(adata, 'total_counts', jitter=0.4, ax=axes[1], show=False)
axes[1].set_title('UMIs per cell')

sc.pl.violin(adata, 'pct_counts_mt', jitter=0.4, ax=axes[2], show=False)
axes[2].set_title('% mitochondrial')

plt.tight_layout()
plt.show()

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt', title='UMIs vs % MT')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', title='UMIs vs Genes')

In [ ]:
# Filter cells: remove low-quality cells and likely doublets
print(f'Before filtering: {adata.n_obs} cells')

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

adata = adata[adata.obs.n_genes_by_counts < 2500, :]
adata = adata[adata.obs.pct_counts_mt < 5, :]

print(f'After filtering:  {adata.n_obs} cells × {adata.n_vars} genes')

## 4. Normalization & Feature Selection

In [ ]:
# Store raw counts for DE testing later
adata.layers['counts'] = adata.X.copy()

# Normalize to 10,000 counts per cell, then log-transform
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Save log-normalized expression for later
adata.raw = adata

# Select highly variable genes
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
print(f'Highly variable genes: {adata.var.highly_variable.sum()}')

sc.pl.highly_variable_genes(adata)

In [ ]:
# Keep only HVGs, regress out confounders, scale
adata = adata[:, adata.var.highly_variable]

sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])
sc.pp.scale(adata, max_value=10)

## 5. Dimensionality Reduction

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50)
plt.title('PCA variance explained')
plt.show()

In [ ]:
# Build neighborhood graph and compute UMAP
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata)
sc.pl.umap(adata, title='UMAP (before clustering)')

## 6. Clustering

In [ ]:
# Leiden clustering (resolution controls granularity: higher = more clusters)
sc.tl.leiden(adata, resolution=0.5, random_state=42)

sc.pl.umap(adata, color='leiden', legend_loc='on data',
           title='Leiden clusters (res=0.5)', frameon=False)

print(f'Number of clusters: {adata.obs.leiden.nunique()}')
print(adata.obs.leiden.value_counts())

## 7. Cell Type Annotation

In [ ]:
# Canonical PBMC marker genes
marker_genes = {
    'CD14+ Mono':    ['CD14', 'LYZ'],
    'CD4 T':         ['IL7R', 'CCR7', 'CD4'],
    'CD8 T':         ['CD8A', 'CD8B'],
    'NK':            ['GNLY', 'NKG7'],
    'B cell':        ['MS4A1', 'CD79A'],
    'Dendritic':     ['FCER1A', 'CST3'],
    'FCGR3A+ Mono':  ['FCGR3A', 'MS4A7'],
    'Platelet':      ['PPBP'],
}

sc.pl.dotplot(
    adata,
    marker_genes,
    groupby='leiden',
    standard_scale='var',
    title='Marker gene expression per cluster'
)

In [ ]:
# Visualize individual markers on UMAP
top_markers = ['CD14', 'IL7R', 'CD8A', 'NKG7', 'MS4A1', 'FCER1A', 'FCGR3A', 'PPBP']
sc.pl.umap(adata, color=top_markers, ncols=4, use_raw=True, vmax='p99')

In [ ]:
# Assign cell type labels to clusters based on marker inspection
# Adjust this mapping after reviewing the dotplot above
cluster_labels = {
    '0': 'CD4 T',
    '1': 'CD14+ Mono',
    '2': 'CD4 T',
    '3': 'B cell',
    '4': 'CD8 T',
    '5': 'NK',
    '6': 'FCGR3A+ Mono',
    '7': 'Dendritic',
    '8': 'Platelet',
}

adata.obs['cell_type'] = adata.obs['leiden'].map(cluster_labels).astype('category')

sc.pl.umap(
    adata, color='cell_type',
    legend_loc='right margin',
    title='PBMC cell types',
    frameon=False
)

## 8. Differential Expression

In [ ]:
# Find marker genes for each cluster vs all others (Wilcoxon rank-sum)
sc.tl.rank_genes_groups(
    adata,
    groupby='leiden',
    method='wilcoxon',
    use_raw=True,
    pts=True,          # fraction of cells expressing each gene
)

sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

In [ ]:
# View top DE genes as a table
de_results = sc.get.rank_genes_groups_df(adata, group=None)
de_results.head(20)

In [ ]:
# Heatmap of top 5 marker genes per cluster
sc.pl.rank_genes_groups_heatmap(
    adata,
    n_genes=5,
    use_raw=True,
    swap_axes=True,
    show_gene_labels=True,
    vmin=-3, vmax=3,
    cmap='RdBu_r'
)

In [ ]:
# Focused pairwise DE: CD4 T vs CD8 T
sc.tl.rank_genes_groups(
    adata,
    groupby='cell_type',
    groups=['CD4 T'],
    reference='CD8 T',
    method='wilcoxon',
    use_raw=True,
    key_added='de_cd4_vs_cd8'
)

cd4_vs_cd8 = sc.get.rank_genes_groups_df(adata, group='CD4 T', key='de_cd4_vs_cd8')
print('Top upregulated in CD4 T vs CD8 T:')
cd4_vs_cd8[cd4_vs_cd8.logfoldchanges > 0].head(10)

## 9. Save Results

In [ ]:
adata.write_h5ad('pbmc3k_analyzed.h5ad')
de_results.to_csv('de_all_clusters.csv', index=False)
print('Saved pbmc3k_analyzed.h5ad and de_all_clusters.csv')